# 05 - Ablation, seed robustness, latency, evidence manifest
* Which feature group drives RCAK? (drop-one / only-one ablation, iso-budget metric)
* Is the gain stable across 5 different article-level splits **and** query samples (retrievers refit each time)?
* What does the controller cost in milliseconds per query?
* SHA-256 manifest of every output file (tamper-evident evidence trail for the thesis).

**Outputs:** `csv/ablation.csv`, `csv/seed_robustness.csv`, `csv/latency.csv`, `MANIFEST.csv`

In [ ]:
# ---- Environment setup (works in Colab AND in a local venv / Jupyter) ----
import os, sys, pathlib, subprocess, zipfile
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "sentence-transformers", "python-docx", "psutil"], check=False)
    if not pathlib.Path("adaptive_rag").exists():
        from google.colab import files
        print("Upload rag_adaptive_project.zip (the project archive you downloaded)")
        up = files.upload(); z = [k for k in up if k.endswith(".zip")][0]
        zipfile.ZipFile(z).extractall("."); 
        root = [p for p in pathlib.Path(".").iterdir() if (p / "adaptive_rag").exists()][0]; os.chdir(root)
    try:
        from google.colab import drive; drive.mount("/content/drive")      # results persist on your Drive
        os.environ["RAG_RESULTS_DIR"] = "/content/drive/MyDrive/rag_adaptive_results"
    except Exception as e:
        os.environ["RAG_RESULTS_DIR"] = os.path.abspath("results")
    BACKEND = "st"          # real neural embeddings on Colab (GPU recommended: Runtime > Change runtime type > T4)
else:
    here = pathlib.Path.cwd()
    root = here.parent if here.name == "notebooks" else here
    sys.path.insert(0, str(root)); os.chdir(root)
    os.environ.setdefault("RAG_RESULTS_DIR", str(root / "results_notebooks"))
    BACKEND = "lsa"         # offline CPU baseline (TF-IDF+SVD). Set to "st" if sentence-transformers can download models.
sys.path.insert(0, os.getcwd())
CFG = dict(backend=BACKEND, st_model="sentence-transformers/all-MiniLM-L6-v2", n_queries=int(os.environ.get("RAG_NQ", 10000)), seed=0)
print("backend:", BACKEND, "| results ->", os.environ["RAG_RESULTS_DIR"])


In [ ]:
import pandas as pd
from adaptive_rag import experiment as E, plots as P
from adaptive_rag.utils import results_dir, write_manifest, log_event
B = E.prepare_cached(CFG)
ab = E.ablation(B, n_boot=300); ab.to_csv(results_dir("csv")/"ablation.csv", index=False); P.fig_ablation(ab); display(ab.round(3))

In [ ]:
lat = E.latency(B); lat.to_csv(results_dir("csv")/"latency.csv", index=False); P.fig_latency(lat); display(lat.round(3))

In [ ]:
sd = E.seeds(CFG, seed_list=(0,1,2,3,4)); sd.to_csv(results_dir("csv")/"seed_robustness.csv", index=False); P.fig_seeds(sd)
display(sd.groupby("strategy")[["mean_k","answer_in_context","fixed_at_same_k","diff"]].agg(["mean","std"]).round(3))

In [ ]:
m = write_manifest(); log_event("notebook05_done"); display(pd.read_csv(m).head(20))